## AI타임즈 뉴스 수집 실습 과제

- [AI타임즈 웹사이트](https://www.aitimes.com/news/articleList.html?view_type=sm)에서 인공지능 관련 뉴스 기사를 수집하여 JSON 파일로 저장하
는 과제입니다.

### 개요
- AI타임즈(www.aitimes.com) 웹사이트에서 최신 AI 관련 뉴스 기사의 제목, 내용, 날짜 등을 수집
- 여러 페이지의 뉴스 목록을 수집
- 수집한 데이터를 구조화하여 JSON 파일로 저장

### 구현 단계
**1.  뉴스 목록 페이지 분석 및 요청**

  - 뉴스 기사 목록 페이지 URL 구조 파악
    - `더보기` 클릭 후 네트워크 분석 추천
  - HTTP 요청 함수 구현

**2. 뉴스 목록에서 기사 정보 추출**

   - BeautifulSoup으로 기사 목록 파싱
   - 제목, 요약, URL, 날짜 등 추출

**3. 기사 상세 내용 수집 [선택]**
   - 각 기사 URL로 접속하여 본문 내용 추출

**4. 데이터 저장 및 분석**
   - 수집한 모든 데이터를 JSON 형식으로 저장
   - 날짜 범위 및 기사 개수 통계 작성

---

### 1.  뉴스 목록 페이지 분석 및 요청

![ai타임즈](../../../images/screenshot%202025-10-29%20오후%201.14.43.png)

In [ ]:
# minimal_aitimes_scraper.py
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime

AJAX_URL = "https://www.aitimes.com/news/ajaxArticlePaging.php"
BASE_URL = "https://www.aitimes.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.aitimes.com/news/articleList.html?view_type=sm",
}

def fetch_page(page: int, list_per_page: int = 20):
    """
    list_per_page: 한 페이지 기사 수
    page_per_page: 페이지네이션에 표시할 페이지 개수 (10이면 “1~10” 버튼)
    page: 현재 페이지 번호
    view_type: sm(작은 리스트형 보기) — 사이트에서 쓰는 값 그대로
    box_idxno: 섹션 구분용인데 대부분 0으로 둬도 됨
    """
    params = {
        "list_per_page": list_per_page,
        "page_per_page": 10,
        "page": page,
        "view_type": "sm",
        "box_idxno": 0,
    }
    r = requests.get(AJAX_URL, params=params, headers=HEADERS, timeout=10)
    r.raise_for_status()
    
    # 서버 응답을 JSON 형태로 바로 파싱해서 반환    
    return r.json()

### 2. 뉴스 목록에서 기사 정보 추출

In [ ]:

def parse_items(data):
    """
    서버에서 받은 JSON → 기사 목록(딕셔너리 리스트)
    JSON → [{title,url,summary,date}]
    """
    items = []
    if not data or data.get("result") != "success":
        return items
    for it in data.get("data", []):
        idxno = it.get("idxno", "")
        title_html = it.get("title", "") or ""
        body_html = it.get("body", "") or ""
        # BeautifulSoup를 이용해 HTML 태그 제거
        title = BeautifulSoup(title_html, "html.parser").get_text().strip()
        body = BeautifulSoup(body_html, "html.parser").get_text().strip()
        url = f"{BASE_URL}/news/articleView.html?idxno={idxno}" if idxno else ""
        date_str = it.get("reg_dt") or it.get("date") or None

        # 200자 요약
        summary = (body[:200] + "...") if len(body) > 200 else body

        items.append({
            "title": title,
            "url": url,
            "summary": summary,
            "date": date_str,  # 원문 보존(간단)
        })
    return items

def crawl(pages: int = 3):
    """
    여러 페이지를 순회하며 parse_items() 반복 실행
    """
    all_items = []
    for p in range(1, pages + 1):
        data = fetch_page(p)    # JSON 데이터 요청
        page_items = parse_items(data)  # JSON을 기사 목록으로 변환
        if not page_items:  # 기사 없으면(빈 페이지면) 종료
            break
        all_items.extend(page_items)    # 결과 누적
    return all_items

def simple_stats(items):
    """
    수집 결과 간단 통계
    """
    dates = [it["date"] for it in items if it.get("date")]
    return {
        "count": len(items),
        "min_date": min(dates) if dates else None,
        "max_date": max(dates) if dates else None,
    }


### 3. 기사 상세 내용 수집 [선택]

### 4. 데이터 저장 및 분석

In [ ]:
def save_json(items, path="aitimes_news.json"):
    payload = {
        "source": "aitimes",
        **simple_stats(items),
        "items": items,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"[OK] {len(items)}건 저장 → {path}")

if __name__ == "__main__":
    items = crawl(pages=3)      # ← 필요한 만큼 페이지 수만 바꿔 쓰면 됨
    save_json(items, "aitimes_news.json")


[OK] 60건 저장 → aitimes_news.json
